# 🏗️ Constructora de Eventos — Taviejito

## Simulador de Psicología Geriátrica

> *Trabajemos en conjunto: tú proporcionas la información del paciente y del evento, y yo la integro adecuadamente en el formato del simulador.*

### 📋 ¿Qué vamos a construir?

1. **Rúbrica de Evaluación** — Marco de 4 dimensiones para calificar las respuestas
2. **Perfil de Paciente** — Datos clínicos, psicosociales y médicos
3. **Evento Conductual** — Situación clínica con opciones alineadas a la rúbrica
4. **Exportación** — A PDF (con rúbrica) y a JSON (para el simulador)

---

### 📐 Rúbrica de Evaluación (Predefinida)

| Dimensión | Peso | Óptimo (+3) | Adecuado (+2) | Parcial (+1) | Contraproducente (-1) |
|:----------|:----:|:-----------|:-------------|:------------|:---------------------|
| **1. Razonamiento Diagnóstico Diferencial** | 30% | Prioriza descarte de causas orgánicas/agudas | Identifica necesidad de descartar pero después de hipótesis psicológicas | Asume causa psicológica sin descarte orgánico | Medicaliza o aplica herramientas en contextos inválidos |
| **2. Selección de Herramientas** | 25% | Escala adecuada al contexto y estado actual | Herramienta válida pero no óptima para el momento | Aplica herramientas sin considerar validez | Pruebas contraindicadas en estado actual |
| **3. Análisis Funcional (ABC)** | 25% | Identifica ABC y vincula con desencadenantes específicos | Registro correcto pero vinculación genérica | Describe conducta sin identificar patrones | Ignora contexto, atribuye solo al diagnóstico |
| **4. Comunicación Terapéutica** | 20% | Integra múltiples fuentes, valida emociones | Información parcial, comunicación poco adaptada | Entrevista cerrada, no valida emociones | Comunicación invalidante o infantilizada |

> **¡Importante!** Cada opción de respuesta en los eventos debe señalar EXPLÍCITAMENTE qué dimensión de la rúbrica aborda y en qué nivel se ubica.

## 🤝 ¿Cómo trabajamos juntos?

```
TÚ (información clínica)  ──►  YO (estructuro con rúbrica)  ──►  NOTEBOKK (procesa y exporta)
```

### Flujo de trabajo:

| Paso | Tú haces | Yo hago |
|:----:|:---------|:--------|
| **1** | Me dices: *"Quiero un paciente con..."* | Edito la celda del **Paso 2** con los datos |
| **2** | Me dices: *"El evento es que..."* | Edito la celda del **Paso 3** y mapeo a la rúbrica |
| **3** | **Ejecutas las celdas** (Run All) | El notebook verifica, genera PDF y JSON |
| **4** | Me dices qué ajustar | Vuelvo a editar y repites desde el paso 3 |

### ¿Qué obtienes?

| Formato | Contenido | Para quién |
|:--------|:----------|:-----------|
| **📄 PDF del evento** | Rúbrica + perfil + evento + opciones con dimensión/nivel + evaluación | Docente (planeación) |
| **📄 PDF del alumno** | Retroalimentación por cada dimensión de la rúbrica con puntuación | Alumno (resultados) |
| **💾 JSON** | Datos estructurados para integrar al simulador Taviejito | Desarrollador |
| **🛠️ Modificación JS** | Código para que el simulador muestre la retroalimentación por rúbrica | Simulador |

> 💡 **¿Empezamos?** Solo dime qué paciente quieres construir y yo preparo las celdas.

In [ ]:
# ⚙️ Configuración Inicial y Bibliotecas

import json
import os
from datetime import datetime
from IPython.display import display, HTML, Markdown

# Intentar importar reportlab (para exportar PDF)
try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import letter, A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch, mm
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak, HRFlowable
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    PDF_DISPONIBLE = True
except ImportError:
    PDF_DISPONIBLE = False
    print("⚠️ ReportLab no está instalado. Para exportar PDF ejecuta: pip install reportlab")
    print("⚠️ Mientras tanto, puedes trabajar con los datos y exportar a JSON.")

# Intentar importar ipywidgets (para inputs interactivos)
try:
    import ipywidgets as widgets
    from ipywidgets import interactive, Layout, VBox, HBox
    INTERACTIVO = True
except ImportError:
    INTERACTIVO = False
    print("⚠️ ipywidgets no está disponible. Usaremos inputs de texto plano.")

print("✅ Entorno listo")
print(f"   📄 PDF: {'Disponible' if PDF_DISPONIBLE else 'No disponible (pip install reportlab)'}")
print(f"   🎮 Interactivo: {'Sí' if INTERACTIVO else 'No'}")
print(f"   📁 Directorio: {os.getcwd()}")

## 🧭 Paso 1: Define la Rúbrica de Evaluación

Esta rúbrica es el **corazón del sistema de puntuación**. Cada opción de respuesta en los eventos debe mapear explícitamente a una de estas dimensiones.

> ⚠️ **IMPORTANTE**: Cuando diseñes las 4 opciones de respuesta de un evento, **CADA OPCIÓN debe señalar claramente**:
> - **Dimensión de la rúbrica** que aborda (ej: "D1 - Razonamiento Diagnóstico Diferencial")
> - **Nivel** en que se ubica (Óptimo/Adecuado/Parcial/Contraproducente)
> - **Justificación breve** vinculada al caso concreto

In [ ]:
# 📋 Almacenar la Rúbrica en datos estructurados

rubric = {
    "dimensiones": [
        {
            "id": "D1",
            "nombre": "Razonamiento Diagnóstico Diferencial",
            "peso": 30,
            "niveles": {
                "optimo": "Prioriza descarte de causas orgánicas/agudas (delirium, dolor, iatrogenia) antes de atribuir conducta a causas psicológicas. Usa signos vitales/meds como pistas de descarte.",
                "adecuado": "Identifica necesidad de descartar lo orgánico, pero lo hace después de explorar hipótesis psicológicas o pasa por alto algún dato clínico relevante.",
                "parcial": "Asume causa psicológica/emocional sin descarte orgánico básico. Sesgo confirmatorio hacia lo conductual ignorando señales de alerta médica.",
                "contraproducente": "Medicaliza prematuramente sin evaluación o aplica herramientas cognitivas/emocionales en contextos inválidos (ej. MoCA en delirium). Riesgo de iatrogenia."
            },
            "evidencia": "CENETEC: Guía Dx y Tx Trastornos Cognitivos (2019); DSM-5-TR: Delirium vs Demencia; Inouye et al.: CAM Validation"
        },
        {
            "id": "D2",
            "nombre": "Selección y Aplicación de Herramientas de Evaluación",
            "peso": 25,
            "niveles": {
                "optimo": "Selecciona escala/instrumento adecuado al contexto y estado actual. Justifica elección basándose en evidencia. Interpreta resultados considerando limitaciones del paciente.",
                "adecuado": "Selecciona herramienta válida pero no la óptima para el momento clínico, o la aplica correctamente pero sin justificación clara.",
                "parcial": "Aplica herramientas mecánicamente sin considerar si el paciente puede responder válidamente, o elige instrumentos poco sensibles para el síntoma.",
                "contraproducente": "Aplica pruebas contraindicadas en estado actual, o interpreta resultados sin contexto clínico. Error técnico grave en administración."
            },
            "evidencia": "Tirapu-Ustárroz: Eval. Neuropsicológica en Adultos Mayores; Yesavage: GDS-15 Validación Mexicana; Cohen-Mansfield: CMAI Manual"
        },
        {
            "id": "D3",
            "nombre": "Análisis Funcional del Comportamiento",
            "peso": 25,
            "niveles": {
                "optimo": "Identifica antecedentes, conducta y consecuentes (ABC). Vincula comportamiento con desencadenantes ambientales/emocionales/sociales específicos, no solo diagnóstico.",
                "adecuado": "Realiza registro conductual correcto pero vinculación con desencadenantes es genérica o incompleta. Falta profundidad en análisis funcional.",
                "parcial": "Describe conducta pero no identifica patrones temporales ni desencadenantes. Descripción superficial sin análisis causal.",
                "contraproducente": "Ignora contexto ambiental/social. Atribuye conducta exclusivamente al diagnóstico ('es la demencia') sin considerar factores modificables."
            },
            "evidencia": "Junta de Extremadura: Guía BPSD (Grupo PIDEX); Feil: Terapia de Validación; APA: Guidelines for Psychological Practice with Older Adults"
        },
        {
            "id": "D4",
            "nombre": "Comunicación Terapéutica y Recolección de Información",
            "peso": 20,
            "niveles": {
                "optimo": "Integra info de múltiples fuentes (paciente, familiar, observación, expediente). Formula preguntas abiertas, valida emociones, adapta comunicación al nivel cognitivo.",
                "adecuado": "Recoge información adecuada pero parcial (solo una fuente). Comunicación correcta pero poco adaptada al estado cognitivo/emocional.",
                "parcial": "Entrevista dirigida/cerrada que limita obtención de info. No valida emociones ni adapta comunicación. Depende excesivamente de terceros.",
                "contraproducente": "Comunicación invalidante, confrontativa o infantilizada. No recoge info relevante o interpreta desde prejuicios. Genera desconfianza/angustia."
            },
            "evidencia": "SGXX: Consenso Evaluación Neuropsicológica en Centros Gerontológicos; Kitwood: Modelo Atención Centrada en Persona; NOM-004-SSA3-2012"
        }
    ]
}

# Mostrar resumen
print("✅ Rúbrica cargada con", len(rubric["dimensiones"]), "dimensiones:")
for d in rubric["dimensiones"]:
    print(f"   {d['id']}: {d['nombre']} ({d['peso']}%)")

## 🧑‍⚕️ Paso 2: Construcción del Perfil del Paciente

**Proporcióname la siguiente información** y yo la integraré en el formato del simulador.

> ℹ️ **Ejecuta la celda de abajo** y sigue las instrucciones. Usaremos variables para construir el perfil paso a paso.

In [ ]:
# 🧑‍⚕️ Captura de Datos del Paciente
# Completa las variables con los datos del paciente

paciente = {
    "id": "",  # ej. "dona_elena"
    "nombre": "",  # ej. "Doña Elena"
    "edad": 0,
    "genero": "",
    "diagnosticos_medicos": [],  # lista
    "medicamentos_actuales": [],  # lista de strings
    "personalidad_premorbida": "",
    "contexto_psicosocial": "",
    "signos_vitales_baseline": {
        "presion_arterial": "",
        "frecuencia_cardiaca": 0,
        "glucosa": 0,
        "temperatura": 0.0,
        "saturacion": 0
    },
    "factores_riesgo": [],
    "nivel_cognitivo": "",
    "movilidad": ""
}

print("╔══════════════════════════════════════════════╗")
print("║   INGRESA LOS DATOS DEL PACIENTE            ║")
print("╚══════════════════════════════════════════════╝")
print()
print("Por favor, edita las variables en la siguiente celda de código.")
print("Ejecuta esta celda y la siguiente para capturar los datos.")

In [ ]:
# ✏️ EDITA AQUÍ LOS DATOS DEL PACIENTE
# Reemplaza los valores de ejemplo con los datos que quieras registrar

paciente["id"] = "don_pedro"  # ID único para el simulador
paciente["nombre"] = "Don Pedro"
paciente["edad"] = 75
paciente["genero"] = "M"
paciente["diagnosticos_medicos"] = [
    "EPOC (GOLD II-III)",
    "Artritis Reumatoide",
    "Hipertensión arterial"
]
paciente["medicamentos_actuales"] = [
    "Salbutamol inhalador (cada 6h PRN)",
    "Prednisona 5mg (diario, recién aumentado a 10mg hace 2 semanas)",
    "Ibuprofeno 400mg (cada 12h, dolor articular)",
    "Enalapril 10mg (diario)"
]
paciente["personalidad_premorbida"] = "Gruñón pero participativo, carácter fuerte, disfrutaba el fútbol y el dominó con amigos, orgulloso de su independencia"
paciente["contexto_psicosocial"] = "Vive con su esposa (cuidadora permanente). Tiene 2 hijos que lo visitan los fines de semana. Recientemente dejó de salir a la calle por 'falta de aire'. Su esposa reporta que 'ya no es el mismo'."
paciente["signos_vitales_baseline"] = {
    "presion_arterial": "125/80",
    "frecuencia_cardiaca": 85,
    "glucosa": 105,
    "temperatura": 36.6,
    "saturacion": 90
}
paciente["factores_riesgo"] = ["hipoxia crónica", "depresión iatrogénica", "caídas", "desnutrición", "efectos adversos de corticoides"]
paciente["nivel_cognitivo"] = "normal (sin deterioro conocido)"
paciente["movilidad"] = "independiente con limitaciones (disnea de esfuerzo)"

print("✅ Datos del paciente registrados:")
print(f"   {paciente['nombre']}, {paciente['edad']} años")
print(f"   Diagnósticos: {len(paciente['diagnosticos_medicos'])} registrados")
print(f"   Medicamentos: {len(paciente['medicamentos_actuales'])} registrados")

## 📖 Paso 3: Define el Evento Conductual

Describe el **cambio conductual o emocional observable** que presenta el paciente.

> 🔑 **Reglas del evento:**
> 1. Presenta un CAMBIO CONDUCTUAL O EMOCIONAL observable (agitación, apatía, agresividad, llanto, desorientación, etc.)
> 2. Incluye datos de signos vitales y medicación COMO PISTAS PARA DESCARTE, no como foco de tratamiento.
> 3. Las 4 opciones deben ser ACCIONES DE EVALUACIÓN PSICOLÓGICA, no intervenciones terapéuticas ni médicas.
> 4. La opción óptima debe seguir: DESCARTAR ORGÁNICO → EVALUAR FUNCIONAL → IDENTIFICAR DESENCADENANTE PSICOSOCIAL

In [ ]:
# 📝 Capturar Datos del Evento Conductual

evento = {
    "id": "",  # ej. "evt_psico_001"
    "titulo": "",
    "descripcion_conducta": "",  # Narrativa del comportamiento observado
    "momento_dia": "",  # "morning", "afternoon", "evening"
    "semanas": [],  # semanas en que ocurre ej. [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18]
    "datos_descarte": {
        "signos_vitales": {},  # valores actuales
        "medicacion_reciente": "",  # qué ha tomado/omitido
        "otros": ""  # sueño, alimentación, dolor
    },
    "opciones": []  # 4 opciones de respuesta
}

print("╔══════════════════════════════════════════════╗")
print("║   INGRESA LOS DATOS DEL EVENTO              ║")
print("╚══════════════════════════════════════════════╝")
print()
print("Completa las variables en la siguiente celda.")

In [ ]:
# ✏️ EDITA AQUÍ LOS DATOS DEL EVENTO CONDUCTUAL

evento["id"] = "evt_psico_002"
evento["titulo"] = "Apatía progresiva con desinterés generalizado"
evento["descripcion_conducta"] = (
    "En las últimas 2-3 semanas, Don Pedro presenta una pérdida progresiva de interés "
    "en todas las actividades que antes disfrutaba: ya no ve el fútbol, no quiere jugar "
    "dominó con sus amigos, pasa la mayor parte del día en cama y ha descuidado su "
    "higiene personal. Su esposa reporta que 'ya no le importa nada' y que ha dejado "
    "de usar su inhalador de salbutamol regularmente. También ha perdido el apetito "
    "(~3 kg en el último mes) y duerme más de 10 horas pero se levanta igualmente fatigado."
)
evento["momento_dia"] = "morning"
evento["semanas"] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

evento["datos_descarte"]["signos_vitales"] = {
    "temperatura": "36.4°C",
    "presion_arterial": "130/85 mmHg",
    "frecuencia_cardiaca": 92,
    "saturacion": 87  # Ha bajado de su baseline (90)
}
evento["datos_descarte"]["medicacion_reciente"] = (
    "Olvidos frecuentes del inhalador. La prednisona fue aumentada de 5mg a 10mg/día "
    "hace 2 semanas por exacerbación de EPOC."
)
evento["datos_descarte"]["otros"] = (
    "Pérdida de apetito sostenida, hiporexia. Duerme 10+ horas pero con sueño no reparador. "
    "Episodios de disnea matutina. Refiere dolor articular generalizado. "
    "Esposa reporta que 'se queda mirando al techo sin moverse' por largos periodos."
)

# Las 4 opciones con su mapeo a la rúbrica
evento["opciones"] = [
    {
        "texto": "Realizar evaluación integral escalonada: 1) Verificar saturación de oxígeno y correlacionar apatía con hipoxia crónica, 2) Revisar perfil de efectos adversos de prednisona (depresión iatrogénica por corticoides), 3) Aplicar GDS-15 para cribado de depresión contextualizando resultados con el estado físico basal del paciente.",
        "correcta": True,
        "dimension": "D1",
        "nivel": "optimo",
        "puntuacion": 3,
        "etiqueta_rubrica": "D1 - Óptimo: Razonamiento Diagnóstico Diferencial",
        "feedback": "El razonamiento es óptimo porque prioriza causas orgánicas reversibles antes de atribuir la apatía a depresión primaria. La hipoxia crónica (SpO2 87%) produce síntomas apáticos idénticos a depresión. Los corticoides (Prednisona 10mg) tienen depresión como efecto adverso conocido. El GDS-15 es útil pero debe interpretarse con cautela pues ítems como 'falta de energía' solapan con EPOC.",
        "modifiers": {"health": 3, "trust": 15, "mind": "OK"}
    },
    {
        "texto": "Aplicar la Escala de Depresión Geriátrica de Yesavage (GDS-15) para diagnosticar depresión mayor e iniciar terapia cognitivo-conductual.",
        "correcta": False,
        "dimension": "D2",
        "nivel": "adecuado",
        "puntuacion": 2,
        "etiqueta_rubrica": "D2 - Adecuado: Selección de Herramientas",
        "feedback": "El GDS-15 es un instrumento validado y apropiado para adultos mayores. Sin embargo, aplicarlo sin antes descartar causas orgánicas (hipoxia, corticoides) es incompleto. Además, varios ítems del GDS (falta de energía, problemas de sueño) tienen comorbilidad con EPOC, lo que puede dar falsos positivos. El razonamiento clínico es adecuado pero no óptimo.",
        "modifiers": {"health": 0, "trust": 5, "mind": "OK"}
    },
    {
        "texto": "Realizar una entrevista clínica con la esposa para explorar si la personalidad 'gruñona' de Don Pedro se ha intensificado con la edad y si la apatía es una manifestación de duelo por la pérdida de su independencia funcional.",
        "correcta": False,
        "dimension": "D3",
        "nivel": "parcial",
        "puntuacion": 1,
        "etiqueta_rubrica": "D3 - Parcial: Análisis Funcional del Comportamiento",
        "feedback": "Explorar el contexto psicosocial es relevante, pero atribuir el cuadro apático exclusivamente a la personalidad o al ajuste emocional ignora señales objetivas: SpO2 descendida (87%), pérdida de peso (3kg), y el aumento reciente de corticoides. Hay riesgo de normalizar la apatía como 'parte de su personalidad' cuando hay causas médicas tratables.",
        "modifiers": {"health": -5, "trust": 5, "mind": "TRISTE"}
    },
    {
        "texto": "Aplicar MoCA (Montreal Cognitive Assessment) y Trail Making Test A y B para descartar demencia frontotemporal, dado que la apatía puede ser un síntoma prodrómico de deterioro del lóbulo frontal.",
        "correcta": False,
        "dimension": "D1",
        "nivel": "contraproducente",
        "puntuacion": -1,
        "etiqueta_rubrica": "D1 - Contraproducente: Razonamiento Diagnóstico",
        "feedback": "Derivar directamente a evaluación de demencia frontotemporal sin descartar causas reversibles es iatrogénico. La hipoxia crónica (SpO2 87%) afecta la perfusión frontal y puede simular deterioro cognitivo de tipo frontal. Además, la fatiga y disnea interfieren con el rendimiento en pruebas neuropsicológicas. Riesgo de etiquetar como demencia lo que es una causa tratable.",
        "modifiers": {"health": -10, "trust": -15, "mind": "ANSIEDAD"}
    }
]

print("✅ Evento registrado:")
print(f"   {evento['titulo']}")
print(f"   {len(evento['opciones'])} opciones de respuesta")
for i, op in enumerate(evento['opciones']):
    print(f"   {chr(65+i)}) [{op['etiqueta_rubrica']}] {'⭐' if op['correcta'] else ''}")

## ✅ Paso 4: Verificación del Evento (Checklist Psicología Geriátrica)

Antes de exportar, verifica que el evento cumpla con estos 5 criterios:

In [ ]:
# ✅ Verificación Automática del Evento

def verificar_evento(paciente, evento, rubric):
    """Verifica que el evento cumpla con los criterios de psicología geriátrica."""
    resultados = []
    errores = []
    
    # Criterio 1: ¿Las opciones son acciones de EVALUACIÓN PSICOLÓGICA?
    es_evaluacion = all(
        any(term in op["texto"].lower() for term in [
            "evaluar", "aplicar", "escala", "entrevista", "observar", 
            "registro", "explorar", "identificar", "medir", "inventario",
            "test", "cuestionario", "abc", "cam", "moca", "yesavage", 
            "lawton", "brody", "gds", "cmai", "minimental", "mmse"
        ])
        for op in evento["opciones"]
    )
    resultados.append(("¿Opciones son de EVALUACIÓN (no tratamiento)?", es_evaluacion))
    if not es_evaluacion:
        errores.append("Alguna opción parece una intervención directa, no una evaluación")
    
    # Criterio 2: ¿Se prioriza el DESCARTE DIFERENCIAL?
    optime = [op for op in evento["opciones"] if op["correcta"]]
    if optime:
        descarte_priorizado = any(
            "descart" in op["texto"].lower() or 
            "delirium" in op["texto"].lower() or
            "cam" in op["texto"].lower() or
            "orgánic" in op["texto"].lower()
            for op in optime
        )
    else:
        descarte_priorizado = False
    resultados.append(("¿Opción óptima prioriza descarte diferencial?", descarte_priorizado))
    if not descarte_priorizado:
        errores.append("La opción óptima debería priorizar descarte de causas orgánicas")
    
    # Criterio 3: ¿Signos vitales funcionan como DATOS DE DESCARTE?
    tiene_datos_descarte = bool(evento["datos_descarte"]["signos_vitales"]) or bool(evento["datos_descarte"]["medicacion_reciente"])
    resultados.append(("¿Hay datos de descarte (signos vitales/meds)?", tiene_datos_descarte))
    if not tiene_datos_descarte:
        errores.append("Faltan datos de signos vitales o medicación para el descarte")
    
    # Criterio 4: ¿Opciones mapean a la rúbrica?
    opciones_con_dimension = all(op.get("dimension") and op.get("nivel") for op in evento["opciones"])
    resultados.append(("¿Opciones mapean a dimensiones de la rúbrica?", opciones_con_dimension))
    if not opciones_con_dimension:
        errores.append("Falta mapeo de dimensiones/niveles de la rúbrica en algunas opciones")
    
    # Criterio 5: ¿Las 4 opciones cubren diferentes niveles?
    niveles_optimo = sum(1 for op in evento["opciones"] if op.get("nivel") == "optimo")
    niveles_contra = sum(1 for op in evento["opciones"] if op.get("nivel") == "contraproducente")
    diversidad_niveles = niveles_optimo == 1 and niveles_contra >= 1
    resultados.append(("¿Diversidad de niveles (1 óptimo, 1+ contraproducente)?", diversidad_niveles))
    if not diversidad_niveles:
        errores.append("Debe haber exactamente 1 óptima y al menos 1 contraproducente")
    
    return resultados, errores

# Ejecutar verificación
resultados, errores = verificar_evento(paciente, evento, rubric)

print("╔══════════════════════════════════════════════╗")
print("║   RESULTADO DE VERIFICACIÓN                 ║")
print("╚══════════════════════════════════════════════╝")
print()
for criterio, cumple in resultados:
    icono = "✅" if cumple else "❌"
    print(f"  {icono} {criterio}")

if errores:
    print(f"\n⚠️  {len(errores)} aspecto(s) a mejorar:")
    for e in errores:
        print(f"   • {e}")
else:
    print(f"\n🎉 ¡Evento listo para exportar!")

## 📤 Paso 5: Exportar Resultados

Puedes exportar en dos formatos:

1. **📄 PDF** — Documento completo con rúbrica + evento + evaluación (para entregar como tarea)
2. **💾 JSON** — Formato para integrar directamente en el simulador Taviejito (`events.json`)

In [ ]:
# 📄 Exportar a PDF con Rúbrica Integrada

def exportar_pdf(paciente, evento, rubric, filename="evento_psicogeriatria.pdf"):
    """Genera un PDF con la rúbrica y el evento completo."""
    
    if not PDF_DISPONIBLE:
        print("❌ ReportLab no está instalado. No se puede generar el PDF.")
        print("   Ejecuta: pip install reportlab")
        return
    
    ruta_pdf = os.path.join(os.getcwd(), filename)
    doc = SimpleDocTemplate(ruta_pdf, pagesize=A4,
                           rightMargin=2*cm, leftMargin=2*cm,
                           topMargin=2*cm, bottomMargin=2*cm)
    
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle('TitleCustom', parent=styles['Title'],
                              fontSize=18, spaceAfter=8, alignment=TA_CENTER))
    styles.add(ParagraphStyle('SubtitleCustom', parent=styles['Normal'],
                              fontSize=11, spaceAfter=4, alignment=TA_CENTER,
                              textColor=colors.HexColor('#555555')))
    styles.add(ParagraphStyle('HeadingCustom', parent=styles['Heading2'],
                              fontSize=13, spaceBefore=12, spaceAfter=6,
                              textColor=colors.HexColor('#1a1a2e')))
    styles.add(ParagraphStyle('BodyCustom', parent=styles['Normal'],
                              fontSize=9, leading=13, alignment=TA_JUSTIFY))
    styles.add(ParagraphStyle('CellStyle', parent=styles['Normal'],
                              fontSize=7.5, leading=9))
    styles.add(ParagraphStyle('SmallCell', parent=styles['Normal'],
                              fontSize=7, leading=8.5))
    
    elementos = []
    
    # ---- PORTADA ----
    elementos.append(Paragraph("TAviejito — Simulador de Psicología Geriátrica", styles['TitleCustom']))
    elementos.append(Paragraph("Constructora de Eventos con Rúbrica de Evaluación", styles['SubtitleCustom']))
    elementos.append(Spacer(1, 0.3*inch))
    
    # ---- SECCIÓN 1: RÚBRICA ----
    elementos.append(Paragraph("A. Rúbrica de Evaluación", styles['HeadingCustom']))
    elementos.append(Spacer(1, 0.1*inch))
    
    # Tabla de rúbrica
    header = [Paragraph("<b>Dimensión</b>", styles['SmallCell']),
              Paragraph("<b>Peso</b>", styles['SmallCell']),
              Paragraph("<b>Óptimo (+3)</b>", styles['SmallCell']),
              Paragraph("<b>Adecuado (+2)</b>", styles['SmallCell']),
              Paragraph("<b>Parcial (+1)</b>", styles['SmallCell']),
              Paragraph("<b>Contraprod. (-1)</b>", styles['SmallCell'])]
    
    tabla_data = [header]
    
    for d in rubric["dimensiones"]:
        fila = [
            Paragraph(f"<b>{d['id']}: {d['nombre']}</b>", styles['SmallCell']),
            Paragraph(f"{d['peso']}%", styles['SmallCell']),
            Paragraph(d['niveles']['optimo'], styles['SmallCell']),
            Paragraph(d['niveles']['adecuado'], styles['SmallCell']),
            Paragraph(d['niveles']['parcial'], styles['SmallCell']),
            Paragraph(d['niveles']['contraproducente'], styles['SmallCell'])
        ]
        tabla_data.append(fila)
    
    # Evidencias
    evidencias_row = [Paragraph("<b>Evidencia Bibliográfica</b>", styles['SmallCell'])]
    for d in rubric["dimensiones"]:
        evidencias_row.append(Paragraph(d['evidencia'], styles['SmallCell']))
    while len(evidencias_row) < 6:
        evidencias_row.append(Paragraph("", styles['SmallCell']))
    tabla_data.append(evidencias_row)
    
    col_widths = [1.5*inch, 0.35*inch, 1.0*inch, 1.0*inch, 1.0*inch, 1.0*inch]
    tabla_rubrica = Table(tabla_data, colWidths=col_widths, repeatRows=1)
    tabla_rubrica.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (1, 0), (1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 7),
        ('GRID', (0, 0), (-1, -2), 0.5, colors.HexColor('#cccccc')),
        ('BACKGROUND', (0, -1), (-1, -1), colors.HexColor('#f0f0f0')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -2), [colors.white, colors.HexColor('#fafafa')]),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
        ('LEFTPADDING', (0, 0), (-1, -1), 3),
        ('RIGHTPADDING', (0, 0), (-1, -1), 3),
    ]))
    
    elementos.append(tabla_rubrica)
    elementos.append(Spacer(1, 0.2*inch))
    
    # ---- SECCIÓN 2: PERFIL DEL PACIENTE ----
    elementos.append(Paragraph("B. Perfil del Paciente", styles['HeadingCustom']))
    
    perfil_data = [
        [Paragraph("<b>Campo</b>", styles['SmallCell']), 
         Paragraph("<b>Valor</b>", styles['SmallCell'])],
        [Paragraph("Nombre", styles['SmallCell']), 
         Paragraph(f"{paciente['nombre']}, {paciente['edad']} años", styles['SmallCell'])],
        [Paragraph("Diagnósticos", styles['SmallCell']), 
         Paragraph(", ".join(paciente['diagnosticos_medicos']), styles['SmallCell'])],
        [Paragraph("Medicamentos", styles['SmallCell']), 
         Paragraph("<br/>".join(paciente['medicamentos_actuales']), styles['SmallCell'])],
        [Paragraph("Personalidad", styles['SmallCell']), 
         Paragraph(paciente['personalidad_premorbida'], styles['SmallCell'])],
        [Paragraph("Contexto psicosocial", styles['SmallCell']), 
         Paragraph(paciente['contexto_psicosocial'], styles['SmallCell'])],
    ]
    
    tabla_perfil = Table(perfil_data, colWidths=[1.5*inch, 4.5*inch])
    tabla_perfil.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#fafafa')]),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
        ('LEFTPADDING', (0, 0), (-1, -1), 4),
    ]))
    elementos.append(tabla_perfil)
    elementos.append(Spacer(1, 0.2*inch))
    
    # ---- SECCIÓN 3: EVENTO ----
    elementos.append(Paragraph("C. Evento Conductual", styles['HeadingCustom']))
    elementos.append(Paragraph(f"<b>{evento['titulo']}</b>", styles['BodyCustom']))
    elementos.append(Spacer(1, 0.1*inch))
    
    elementos.append(Paragraph(f"<b>Descripción:</b> {evento['descripcion_conducta']}", styles['BodyCustom']))
    elementos.append(Spacer(1, 0.1*inch))
    
    elementos.append(Paragraph("<b>Datos Clínicos de Descarte:</b>", styles['BodyCustom']))
    for k, v in evento['datos_descarte']['signos_vitales'].items():
        elementos.append(Paragraph(f"• {k}: {v}", styles['BodyCustom']))
    elementos.append(Paragraph(f"• Medicación reciente: {evento['datos_descarte']['medicacion_reciente']}", styles['BodyCustom']))
    elementos.append(Paragraph(f"• Otros: {evento['datos_descarte']['otros']}", styles['BodyCustom']))
    elementos.append(Spacer(1, 0.15*inch))
    
    # ---- SECCIÓN 4: OPCIONES ----
    elementos.append(Paragraph("D. Opciones de Respuesta", styles['HeadingCustom']))
    
    letras = ['A', 'B', 'C', 'D']
    for i, op in enumerate(evento['opciones']):
        color_nivel = {
            'optimo': '#1b5e20',
            'adecuado': '#f57f17',
            'parcial': '#e65100',
            'contraproducente': '#b71c1c'
        }
        etiqueta_nivel = {
            'optimo': 'ÓPTIMA (+3)',
            'adecuado': 'ADECUADA (+2)',
            'parcial': 'PARCIAL (+1)',
            'contraproducente': 'CONTRAPRODUCENTE (-1)'
        }
        color = color_nivel.get(op['nivel'], '#333333')
        
        opt_text = f"⭐ {etiqueta_nivel.get(op['nivel'], '')}" if op['correcta'] else etiqueta_nivel.get(op['nivel'], '')
        
        elementos.append(Paragraph(
            f"<b>{letras[i]})</b> [{op['etiqueta_rubrica']}] <font color='{color}'><b>{opt_text}</b></font>",
            styles['BodyCustom']
        ))
        elementos.append(Paragraph(op['texto'], styles['BodyCustom']))
        elementos.append(Paragraph(f"<i>Feedback: {op['feedback']}</i>", styles['BodyCustom']))
        elementos.append(Spacer(1, 0.08*inch))
    
    # ---- SECCIÓN 5: EVALUACIÓN ----
    elementos.append(Paragraph("E. Evaluación (basada en la rúbrica)", styles['HeadingCustom']))
    
    eval_data = [[Paragraph("<b>Nivel</b>", styles['SmallCell']),
                  Paragraph("<b>Opción</b>", styles['SmallCell']),
                  Paragraph("<b>Dimensión</b>", styles['SmallCell']),
                  Paragraph("<b>Justificación</b>", styles['SmallCell'])]]
    
    for i, op in enumerate(evento['opciones']):
        eval_data.append([
            Paragraph(etiqueta_nivel.get(op['nivel'], ''), styles['SmallCell']),
            Paragraph(letras[i], styles['SmallCell']),
            Paragraph(op.get('dimension', ''), styles['SmallCell']),
            Paragraph(op['feedback'], styles['SmallCell'])
        ])
    
    tabla_eval = Table(eval_data, colWidths=[1.2*inch, 0.4*inch, 0.6*inch, 3.8*inch])
    tabla_eval.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#fafafa')]),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
        ('LEFTPADDING', (0, 0), (-1, -1), 4),
    ]))
    elementos.append(tabla_eval)
    elementos.append(Spacer(1, 0.2*inch))
    
    # ---- SECCIÓN 6: REFERENCIAS ----
    elementos.append(Paragraph("F. Referencias Bibliográficas", styles['HeadingCustom']))
    for d in rubric["dimensiones"]:
        elementos.append(Paragraph(f"<b>{d['id']}: {d['nombre']}</b>", styles['BodyCustom']))
        elementos.append(Paragraph(f"  {d['evidencia']}", styles['BodyCustom']))
        elementos.append(Spacer(1, 0.05*inch))
    
    elementos.append(Spacer(1, 0.3*inch))
    elementos.append(Paragraph(f"Documento generado el {datetime.now().strftime('%d/%m/%Y %H:%M')} — Taviejito Simulator",
                              styles['SubtitleCustom']))
    
    doc.build(elementos)
    print(f"✅ PDF exportado: {ruta_pdf}")

# Exportar
exportar_pdf(paciente, evento, rubric, "evento_psicogeriatria.pdf")

In [ ]:
# 💾 Exportar a JSON (formato Taviejito)

def exportar_json_taviejito(paciente, evento, filename="evento_taviejito.json"):
    """
    Exporta el evento en el formato que usa el simulador Taviejito.
    Estructura compatible con events.json y patients.json
    """
    
    # Construir ID único para el paciente
    patient_id = paciente["id"] if paciente["id"] else f"patient_{hash(paciente['nombre'])}"
    
    # Datos del paciente en formato Taviejito
    patient_data = {
        "id": patient_id,
        "name": paciente["nombre"],
        "age": paciente["edad"],
        "gender": paciente["genero"],
        "conditions": paciente["diagnosticos_medicos"],
        "medications": [{"name": m, "schedule": "08:00", "taken": False} for m in paciente["medicamentos_actuales"]],
        "personality": paciente.get("personalidad_premorbida", "").split(",")[0].lower() if paciente.get("personalidad_premorbida") else "amable",
        "baselineVitals": {
            "bloodPressure": paciente["signos_vitales_baseline"].get("presion_arterial", "120/80"),
            "heartRate": paciente["signos_vitales_baseline"].get("frecuencia_cardiaca", 72),
            "glucose": paciente["signos_vitales_baseline"].get("glucosa", 100),
            "temperature": paciente["signos_vitales_baseline"].get("temperatura", 36.5),
            "oxygenSat": paciente["signos_vitales_baseline"].get("saturacion", 97)
        },
        "riskFactors": paciente.get("factores_riesgo", []),
        "spriteId": patient_id
    }
    
    # Construir opciones en formato Taviejito
    opciones_taviejito = []
    for op in evento["opciones"]:
        opciones_taviejito.append({
            "text": op["texto"],
            "correct": op["correcta"],
            "feedback": f"[{op['etiqueta_rubrica']}] {op['feedback']}",
            "modifiers": op.get("modifiers", {}),
            "icon": "⭐" if op["correcta"] else ("💚" if op["nivel"] == "optimo" else "⚠️" if op["nivel"] != "contraproducente" else "💔")
        })
    
    # Evento en formato Taviejito
    event_data = {
        "id": evento["id"],
        "category": "psicologia",
        "description": f"{evento['titulo']}. {evento['descripcion_conducta']}",
        "patientConditions": paciente["diagnosticos_medicos"],
        "requiresCheck": ["vitals", "mind"],
        "timeOfDay": evento.get("momento_dia", "afternoon"),
        "weeks": evento.get("semanas", [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18]),
        "options": opciones_taviejito,
        # Metadatos de la rúbrica (para exportación PDF)
        "_rubric_metadata": {
            "datos_descarte": evento["datos_descarte"],
            "evaluacion_opciones": [
                {
                    "letra": chr(65+i),
                    "dimension": op.get("dimension", ""),
                    "nivel": op.get("nivel", ""),
                    "puntuacion": op.get("puntuacion", 0),
                    "feedback": op["feedback"]
                }
                for i, op in enumerate(evento["opciones"])
            ]
        }
    }
    
    # Guardar JSON
    ruta_json = os.path.join(os.getcwd(), filename)
    with open(ruta_json, 'w', encoding='utf-8') as f:
        json.dump({
            "event": event_data,
            "patient": patient_data
        }, f, ensure_ascii=False, indent=2)
    
    print(f"✅ JSON exportado: {ruta_json}")
    print(f"   Paciente: {patient_data['name']}")
    print(f"   Evento: {evento['id']} — {evento['titulo']}")
    print(f"   Opciones: {len(opciones_taviejito)}")
    print(f"\n📌 Para integrar en el simulador:")
    print(f"   1. Copia estos datos a data/events.json o data/patients.json")
    print(f"   2. O edita directamente eventsystem.js para agregar el evento")
    
    return event_data, patient_data

# Exportar
event_data, patient_data = exportar_json_taviejito(paciente, evento, "evento_taviejito.json")

## 🧑‍🎓 Paso 6: Generar Reporte del Alumno (PDF con rúbrica)

Cuando un alumno responde en el simulador, en lugar de ver solo "✅ CORRECTO" o "❌ ERROR", debe ver su **retroalimentación organizada por dimensión de la rúbrica**.

> 📌 **¿Cómo funciona?**
> - Cada opción elegida se traduce a: *Dimensión + Nivel + Puntuación + Feedback*
> - El PDF del alumno muestra: qué dimensión evaluó, en qué nivel, por qué, y qué referencia lo respalda
> - Ya no es "acertaste/fallaste" — es **"tu razonamiento fue D1-óptimo porque priorizaste el descarte orgánico"**

In [ ]:
# 🧑‍🎓 Generar Reporte de Resultados para el Alumno (basado en rúbrica)

def generar_reporte_alumno(respuestas_alumno, paciente, rubric, filename="reporte_alumno_psicogeriatria.pdf"):
    """
    Genera un PDF individual para el alumno con su retroalimentación
    organizada por dimensión de la rúbrica.
    
    respuestas_alumno: lista de dicts con:
        - evento_id: str
        - evento_titulo: str
        - opcion_elegida: str (A/B/C/D)
        - opcion_texto: str
        - dimension: str (D1, D2, D3, D4)
        - nivel: str (optimo/adecuado/parcial/contraproducente)
        - puntuacion: int (+3/+2/+1/-1)
        - feedback: str
    """
    if not PDF_DISPONIBLE:
        print("❌ ReportLab no está instalado. pip install reportlab")
        return
    
    ruta_pdf = os.path.join(os.getcwd(), filename)
    doc = SimpleDocTemplate(ruta_pdf, pagesize=A4,
                           rightMargin=2*cm, leftMargin=2*cm,
                           topMargin=2*cm, bottomMargin=2*cm)
    
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle('TitleCustom', parent=styles['Title'],
                              fontSize=16, spaceAfter=6, alignment=TA_CENTER))
    styles.add(ParagraphStyle('SubtitleCustom', parent=styles['Normal'],
                              fontSize=10, spaceAfter=4, alignment=TA_CENTER,
                              textColor=colors.HexColor('#555555')))
    styles.add(ParagraphStyle('HeadingCustom', parent=styles['Heading2'],
                              fontSize=12, spaceBefore=10, spaceAfter=4,
                              textColor=colors.HexColor('#1a1a2e')))
    styles.add(ParagraphStyle('BodyCustom', parent=styles['Normal'],
                              fontSize=9, leading=13, alignment=TA_JUSTIFY))
    styles.add(ParagraphStyle('CellStyle', parent=styles['Normal'],
                              fontSize=8, leading=10))
    styles.add(ParagraphStyle('FeedbackStyle', parent=styles['Normal'],
                              fontSize=8, leading=10,
                              leftIndent=10, textColor=colors.HexColor('#2c3e50')))
    
    elementos = []
    
    # PORTADA
    elementos.append(Paragraph("Reporte de Evaluación — Psicología Geriátrica", styles['TitleCustom']))
    elementos.append(Paragraph("TAviejito Simulator | Psychenudo Puebla", styles['SubtitleCustom']))
    elementos.append(Spacer(1, 0.2*inch))
    
    # DATOS DEL ALUMNO
    elementos.append(Paragraph("Datos del Estudiante", styles['HeadingCustom']))
    
    alumno_data = [
        [Paragraph("<b>Campo</b>", styles['CellStyle']),
         Paragraph("<b>Valor</b>", styles['CellStyle'])],
        [Paragraph("Nombre", styles['CellStyle']),
         Paragraph("[Nombre del alumno]", styles['CellStyle'])],
        [Paragraph("Matrícula", styles['CellStyle']),
         Paragraph("[Matrícula]", styles['CellStyle'])],
        [Paragraph("Paciente asignado", styles['CellStyle']),
         Paragraph(f"{paciente['nombre']} ({paciente['edad']} años)", styles['CellStyle'])],
        [Paragraph("Diagnósticos", styles['CellStyle']),
         Paragraph(", ".join(paciente['diagnosticos_medicos']), styles['CellStyle'])],
    ]
    
    tabla_alumno = Table(alumno_data, colWidths=[1.5*inch, 4.5*inch])
    tabla_alumno.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#fafafa')]),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
        ('LEFTPADDING', (0, 0), (-1, -1), 4),
    ]))
    elementos.append(tabla_alumno)
    elementos.append(Spacer(1, 0.2*inch))
    
    # RÚBRICA DE REFERENCIA
    elementos.append(Paragraph("Rúbrica de Referencia", styles['HeadingCustom']))
    
    rubric_header = [
        Paragraph("<b>Dimensión</b>", styles['CellStyle']),
        Paragraph("<b>Peso</b>", styles['CellStyle']),
        Paragraph("<b>Tu nivel</b>", styles['CellStyle']),
        Paragraph("<b>Puntos</b>", styles['CellStyle']),
    ]
    rubric_rows = [rubric_header]
    
    # Calcular puntuación total
    puntuacion_total = sum(r.get('puntuacion', 0) for r in respuestas_alumno)
    max_posible = len(respuestas_alumno) * 3
    
    for d in rubric["dimensiones"]:
        # Buscar respuestas del alumno para esta dimensión
        resp_dim = [r for r in respuestas_alumno if r.get('dimension') == d['id']]
        if resp_dim:
            nivel_obtenido = resp_dim[0].get('nivel', 'no evaluado')
            puntos = resp_dim[0].get('puntuacion', 0)
        else:
            nivel_obtenido = 'no evaluado'
            puntos = '-'
        
        nombre_nivel = {
            'optimo': 'Óptimo (+3)',
            'adecuado': 'Adecuado (+2)',
            'parcial': 'Parcial (+1)',
            'contraproducente': 'Contraproducente (-1)',
            'no evaluado': '—'
        }
        
        rubric_rows.append([
            Paragraph(f"<b>{d['id']}: {d['nombre']}</b>", styles['CellStyle']),
            Paragraph(f"{d['peso']}%", styles['CellStyle']),
            Paragraph(nombre_nivel.get(nivel_obtenido, '—'), styles['CellStyle']),
            Paragraph(str(puntos) if puntos != '-' else '—', styles['CellStyle']),
        ])
    
    # Fila de total
    pct = round((puntuacion_total / max_posible) * 100, 1) if max_posible > 0 else 0
    rubric_rows.append([
        Paragraph("<b>TOTAL</b>", styles['CellStyle']),
        Paragraph("<b>100%</b>", styles['CellStyle']),
        Paragraph(f"<b>{puntuacion_total}/{max_posible}</b>", styles['CellStyle']),
        Paragraph(f"<b>{pct}%</b>", styles['CellStyle']),
    ])
    
    tabla_rubric = Table(rubric_rows, colWidths=[2.0*inch, 0.6*inch, 1.5*inch, 0.6*inch])
    tabla_rubric.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (1, 0), (-1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('ROWBACKGROUNDS', (0, 1), (-1, -2), [colors.white, colors.HexColor('#fafafa')]),
        ('BACKGROUND', (0, -1), (-1, -1), colors.HexColor('#e8eaf6')),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
    ]))
    elementos.append(tabla_rubric)
    elementos.append(Spacer(1, 0.2*inch))
    
    # RETROALIMENTACIÓN POR EVENTO
    elementos.append(Paragraph("Retroalimentación Detallada", styles['HeadingCustom']))
    
    colores_nivel = {
        'optimo': colors.HexColor('#1b5e20'),
        'adecuado': colors.HexColor('#f57f17'),
        'parcial': colors.HexColor('#e65100'),
        'contraproducente': colors.HexColor('#b71c1c'),
    }
    iconos_nivel = {
        'optimo': '🟢 ÓPTIMO (+3)',
        'adecuado': '🟡 ADECUADO (+2)',
        'parcial': '🟠 PARCIAL (+1)',
        'contraproducente': '🔴 CONTRAPRODUCENTE (-1)',
    }
    
    for i, respuesta in enumerate(respuestas_alumno):
        color = colores_nivel.get(respuesta.get('nivel', ''), colors.HexColor('#333'))
        icono = iconos_nivel.get(respuesta.get('nivel', ''), '')
        
        elementos.append(Paragraph(
            f"<b>Evento {i+1}: {respuesta.get('evento_titulo', 'Sin título')}</b>",
            styles['BodyCustom']
        ))
        elementos.append(Spacer(1, 0.05*inch))
        
        # Cabecera con dimensión y nivel
        header_text = (
            f"<font color='{color.hexval()}'><b>{icono}</b></font> &nbsp;|&nbsp; "
            f"<b>Dimensión:</b> {respuesta.get('dimension', '?')} — "
            f"{respuesta.get('dimension_nombre', '')} &nbsp;|&nbsp; "
            f"<b>Puntuación:</b> {respuesta.get('puntuacion', 0)} pts"
        )
        elementos.append(Paragraph(header_text, styles['FeedbackStyle']))
        elementos.append(Spacer(1, 0.03*inch))
        
        # Opción elegida
        elementos.append(Paragraph(
            f"<b>Tu respuesta:</b> {respuesta.get('opcion_elegida', '?')}) "
            f"{respuesta.get('opcion_texto', '')}",
            styles['FeedbackStyle']
        ))
        elementos.append(Spacer(1, 0.03*inch))
        
        # Feedback
        elementos.append(Paragraph(
            f"<b>Retroalimentación:</b> {respuesta.get('feedback', '')}",
            styles['FeedbackStyle']
        ))
        
        # Referencia
        ref = respuesta.get('referencia', '')
        if ref:
            elementos.append(Paragraph(
                f"<i>Referencia: {ref}</i>",
                styles['FeedbackStyle']
            ))
        
        elementos.append(Spacer(1, 0.12*inch))
    
    # Resumen final
    elementos.append(Spacer(1, 0.15*inch))
    elementos.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor('#cccccc')))
    elementos.append(Spacer(1, 0.1*inch))
    
    if pct >= 75:
        conclusion = "🎉 Desempeño destacado. Priorizaste correctamente el descarte diferencial."
    elif pct >= 50:
        conclusion = "👍 Desempeño adecuado. Revisa las oportunidades de mejora en las dimensiones con menor puntuación."
    elif pct >= 25:
        conclusion = "⚠️ Desempeño en desarrollo. Es fundamental reforzar el razonamiento diagnóstico diferencial."
    else:
        conclusion = "🔴 Es necesario revisar los fundamentos de la evaluación en psicología geriátrica."
    
    elementos.append(Paragraph(f"<b>Conclusión:</b> {conclusion}", styles['BodyCustom']))
    elementos.append(Spacer(1, 0.15*inch))
    elementos.append(Paragraph(
        f"Reporte generado el {datetime.now().strftime('%d/%m/%Y %H:%M')} — TAviejito Simulator",
        styles['SubtitleCustom']
    ))
    
    doc.build(elementos)
    print(f"✅ Reporte del alumno exportado: {ruta_pdf}")
    print(f"   Puntuación total: {puntuacion_total}/{max_posible} ({pct}%)")
    return ruta_pdf


# ── EJEMPLO DE USO ──────────────────────────────────────────────────
# Simula las respuestas de un alumno para probar el reporte
respuestas_ejemplo = [
    {
        "evento_id": "evt_psico_001",
        "evento_titulo": "Agitación psicomotriz y alucinaciones",
        "opcion_elegida": "B",
        "opcion_texto": "Aplicar CAM y registro ABC para descartar delirium",
        "dimension": "D1",
        "dimension_nombre": "Razonamiento Diagnóstico Diferencial",
        "nivel": "optimo",
        "puntuacion": 3,
        "feedback": "Priorizaste el descarte de causa orgánica aguda. La fluctuación de conciencia + fiebre + disuria sugerían Delirium por ITU. Correcto uso del CAM (Inouye).",
        "referencia": "Inouye et al.: CAM Validation Study; DSM-5-TR: Delirium vs Demencia"
    }
]

print("🧪 Ejecuta esta celda para generar un reporte de prueba.")
print("   Luego, en el Paso 2 puedes sustituir 'respuestas_ejemplo' con las respuestas reales del alumno.")
print()

if PDF_DISPONIBLE:
    generar_reporte_alumno(respuestas_ejemplo, paciente, rubric, "reporte_alumno_ejemplo.pdf")
else:
    print("❌ ReportLab no instalado. pip install reportlab")

## 🛠️ Paso 7: Integrar en el Simulador Taviejito

Para que el simulador muestre la retroalimentación **por rúbrica** (no solo ✅/❌), necesitamos modificar 2 archivos:

### 📁 `data/events.json` (o `eventsystem.js`)
Agregar a cada opción los campos:
- `"dimension": "D1"` — qué dimensión evalúa
- `"nivel": "optimo"` — nivel en la rúbrica
- `"puntuacion": 3` — puntos asociados
- `"etiqueta_rubrica": "D1 - Óptimo: Razonamiento Diagnóstico Diferencial"`

### 📁 `js/ui.js`
Modificar el panel de feedback para que muestre:
```
┌─────────────────────────────────────┐
│ 🟢 D1 — ÓPTIMO (+3)                │
│ Dimensión: Razonamiento Diferencial │
│                                     │
│ Priorizaste descarte de delirium    │
│ usando CAM.                         │
│                                     │
│ Referencia: Inouye et al.           │
└─────────────────────────────────────┘
```

> **La celda de abajo** genera el código JavaScript listo para copiar/pegar.

In [ ]:
# 🛠️ Generar código JavaScript para el simulador (feedback por rúbrica)

def generar_js_simulador(evento):
    """
    Genera el fragmento de código JavaScript para modificar el simulador
    y que muestre la retroalimentación basada en la rúbrica.
    """
    
    mapeo_dimensiones = {
        "D1": "Razonamiento Diagnóstico Diferencial",
        "D2": "Selección y Aplicación de Herramientas",
        "D3": "Análisis Funcional del Comportamiento",
        "D4": "Comunicación Terapéutica"
    }
    
    pesos = {"D1": "30%", "D2": "25%", "D3": "25%", "D4": "20%"}
    
    etiqueta_nivel = {
        "optimo": "ÓPTIMO (+3)",
        "adecuado": "ADECUADO (+2)",
        "parcial": "PARCIAL (+1)",
        "contraproducente": "CONTRAPRODUCENTE (-1)"
    }
    
    color_nivel = {
        "optimo": "#1b5e20",
        "adecuado": "#f57f17",
        "parcial": "#e65100",
        "contraproducente": "#b71c1c"
    }
    
    icono_nivel = {
        "optimo": "🟢",
        "adecuado": "🟡",
        "parcial": "🟠",
        "contraproducente": "🔴"
    }
    
    # Generar opciones para events.json
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║   🛠️  CÓDIGO PARA INTEGRAR EN EL SIMULADOR                ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print()
    print("📁 1. COPIA ESTO EN data/events.json (estructura con rúbrica):")
    print()
    
    opciones_json = []
    letras = ['A', 'B', 'C', 'D']
    for i, op in enumerate(evento['opciones']):
        op_json = {
            "text": op['texto'],
            "correct": op['correcta'],
            "dimension": op['dimension'],
            "nivel": op['nivel'],
            "puntuacion": op['puntuacion'],
            "etiqueta_rubrica": op['etiqueta_rubrica'],
            "feedback": op['feedback'],
            "modifiers": op.get('modifiers', {}),
            "icon": icono_nivel.get(op['nivel'], '💚')
        }
        opciones_json.append(op_json)
    
    print("    [")
    for i, op in enumerate(evento['opciones']):
        comma = "," if i < len(evento['opciones']) - 1 else ""
        print(f'      {{')
        print(f'        "text": "{op["texto"][:70]}...",')
        print(f'        "correct": {"true" if op["correcta"] else "false"},')
        print(f'        "dimension": "{op["dimension"]}",')
        print(f'        "nivel": "{op["nivel"]}",')
        print(f'        "puntuacion": {op["puntuacion"]},')
        print(f'        "etiqueta_rubrica": "{op["etiqueta_rubrica"]}",')
        print(f'        "feedback": "{op["feedback"][:80]}..."')
        print(f'      }}{comma}')
    print("    ]")
    print()
    
    # Generar código para modificar el feedback UI
    print()
    print("📁 2. COPIA ESTO EN js/ui.js (o donde se muestre el feedback):")
    print()
    print('''    // =====================================================
    // Código para mostrar feedback por RÚBRICA
    // Reemplazar la función showFeedback() actual
    // =====================================================
    
    function showRubricFeedback(option, isCorrect) {
        // Determinar nivel y dimensión desde la opción
        const niveles = {
            'optimo': { label: 'ÓPTIMO (+3)', color: '#1b5e20', icono: '🟢' },
            'adecuado': { label: 'ADECUADO (+2)', color: '#f57f17', icono: '🟡' },
            'parcial': { label: 'PARCIAL (+1)', color: '#e65100', icono: '🟠' },
            'contraproducente': { label: 'CONTRAPRODUCENTE (-1)', color: '#b71c1c', icono: '🔴' }
        };
        
        const dimensiones = {
            'D1': 'Razonamiento Diagnóstico Diferencial (30%)',
            'D2': 'Selección y Aplicación de Herramientas (25%)',
            'D3': 'Análisis Funcional del Comportamiento (25%)',
            'D4': 'Comunicación Terapéutica y Recolección (20%)'
        };
        
        const nivel = option.nivel || (isCorrect ? 'optimo' : 'contraproducente');
        const dim = option.dimension || 'D1';
        const info = niveles[nivel] || niveles.optimo;
        const dimNombre = dimensiones[dim] || dim;
        const puntuacion = option.puntuacion || (isCorrect ? 3 : -1);
        
        // Construir HTML de feedback
        const feedbackHTML = `
            <div style="border-left: 4px solid ${info.color}; padding: 8px 12px; margin: 10px 0; background: #f8f9fa; border-radius: 4px;">
                <div style="font-size: 14px; font-weight: bold; color: ${info.color};">
                    ${info.icono} ${dim} — ${info.label}
                </div>
                <div style="font-size: 11px; color: #666; margin: 4px 0;">
                    ${dimNombre} | Puntuación: ${puntuacion} pts
                </div>
                <div style="font-size: 13px; margin: 8px 0; color: #333;">
                    ${option.feedback || ''}
                </div>
            </div>
        `;
        
        return feedbackHTML;
    }''')
    
    print()
    print()
    print("📁 3. Para INTEGRAR en eventsystem.js, usa el JSON exportado:")
    print('   - Los eventos ahora incluyen: dimension, nivel, puntuacion, etiqueta_rubrica')
    print('   - Se mantienen los campos "correct" y "modifiers" para compatibilidad')
    print()

# Mostrar el código
generar_js_simulador(evento)

In [ ]:
# 📊 Vista General del Evento Construido

print("╔══════════════════════════════════════════════════════════════╗")
print("║           🏗️  CONSTRUCTORA DE EVENTOS — TAVIEJITO            ║")
print("╚══════════════════════════════════════════════════════════════╝")
print()
print(f"  📋 PACIENTE: {paciente['nombre']} ({paciente['edad']} años)")
print(f"  🏥 Diagnósticos: {', '.join(paciente['diagnosticos_medicos'])}")
print(f"  💊 Medicamentos: {', '.join(paciente['medicamentos_actuales'][:3])}")
print(f"  🧠 Personalidad: {paciente['personalidad_premorbida'][:70]}...")
print(f"  👨‍👩‍👧 Contexto: {paciente['contexto_psicosocial'][:80]}...")
print()
print("─" * 60)
print(f"  📖 EVENTO: {evento['titulo']}")
print(f"  🕒 Momento: {evento['momento_dia']}")
print(f"  📝 {evento['descripcion_conducta'][:120]}...")
print()
print("  🔍 DATOS DE DESCARTE:")
for k, v in evento['datos_descarte']['signos_vitales'].items():
    print(f"     • {k}: {v}")
print(f"     • Medicación: {evento['datos_descarte']['medicacion_reciente']}")
print(f"     • Otros: {evento['datos_descarte']['otros'][:80]}...")
print()
print("  📋 OPCIONES DE RESPUESTA (mapeadas a rúbrica):")
letras = ['A', 'B', 'C', 'D']
nombres_nivel = {'optimo': 'ÓPTIMO (+3)', 'adecuado': 'ADECUADO (+2)', 'parcial': 'PARCIAL (+1)', 'contraproducente': 'CONTRAPRODUCENTE (-1)'}
for i, op in enumerate(evento['opciones']):
    estrella = "⭐ " if op['correcta'] else "   "
    print(f"     {estrella}{letras[i]}) [{op['dimension']}] {nombres_nivel[op['nivel']]}")
    print(f"         {op['texto'][:90]}...")
print()
print("─" * 60)
print(f"  📄 PDF: evento_psicogeriatria.pdf ({'✅ listo' if PDF_DISPONIBLE else '❌ instala reportlab'})")
print(f"  💾 JSON: evento_taviejito.json (✅ listo)")
print()
print("╚══════════════════════════════════════════════════════════════╝")

## 🚀 Cómo usar este notebook

### Para construir un NUEVO evento:

1. **Ve a la celda "✏️ EDITA AQUÍ LOS DATOS DEL PACIENTE"** (Paso 2) y modifica los datos con el nuevo perfil
2. **Ve a la celda "✏️ EDITA AQUÍ LOS DATOS DEL EVENTO CONDUCTUAL"** (Paso 3) y describe el nuevo evento
3. **Ejecuta todas las celdas en orden** (Cell → Run All)
4. **Revisa la verificación automática** (Paso 4)
5. **Exporta** a PDF (para entregar) y JSON (para el simulador)

### 📋 Lo que NECESITO que me proporciones para cada evento:

| Elemento | ¿Qué necesito? | Ejemplo |
|:---------|:---------------|:--------|
| **Perfil del paciente** | Nombre, edad, diagnósticos, medicamentos, personalidad, contexto psicosocial | Doña Elena, 76, HTA+DM2+DCL |
| **Conducta problema** | ¿Qué hace? ¿Desde cuándo? ¿En qué contexto? ¿Con qué intensidad? | Agitación + alucinaciones + agresividad |
| **Datos de descarte** | Signos vitales actuales, medicación reciente, otros síntomas | Fiebre 37.9°C, disuria, insomnio |
| **Opción óptima** | ¿Qué evaluación permite descartar lo orgánico primero? | CAM + registro ABC |
| **Opciones alternativas** | Una adecuada, una parcial y una contraproducente | Lawton (adecuada), Yesavage (parcial), MoCA (contraproducente) |

### 📌 Importante para exportar PDF

```bash
pip install reportlab
```

---

> **¿Listo para el siguiente evento?** Solo edita los datos de las celdas de los Pasos 2 y 3, y vuelve a ejecutar. Yo integraré todo en el formato correcto.